# Fact-Checker Agent: ReAct + Reflexion

**Exercise objective:** build an agent that combines two agentic paradigms - **ReAct** (reason / act / observe over tool calls) and **Reflexion** (self-critique with memory across iterations) - to verify factual claims with a structured verdict and traceable sources.

## Where this sits in the course

Module 03 (*Paradigmi Agentici*) introduces three reasoning paradigms, one per practice notebook:

| Paradigm | Idea | Practice notebook |
|----------|------|-------------------|
| ReAct | Interleave reasoning steps with tool calls | `01 02 ReAct.ipynb` |
| Plan & Execute | Plan the full sequence first, then execute it | `03 Plan Execute.ipynb` |
| Reflexion | Critique the output, retry with the critique in memory | `04 Reflexion.ipynb` |

This exercise combines ReAct (inner loop) with Reflexion (outer loop). Plan & Execute is the third paradigm in the module; it is orthogonal to what we build here and would be a natural extension if the task required up-front planning of which sources to consult.

## Comparison with previous exercises

| Aspect | Ex. 01 (workflow) | Ex. 02 (tool agent) | Ex. 03 (this) |
|--------|-------------------|---------------------|----------------|
| Control flow | Python dispatch | LLM tool calling | LLM tool calling + self-critique loop |
| LLM calls per question | 2 | 4-8 | 8-20+ |
| Memory of own mistakes | None | None | Critiques carry over across Reflexion iterations |
| Verdict structure | One-line prose | Free-form natural language | Structured (verdict, reasoning, sources, confidence) |

## Architecture

```
claim
  └─[ReAct agent]── reason → tool call → observe → ... → verdict v1
        │                                                     │
        ▼                                                     ▼
    trace of (tool, args, observation) ──────────────► [Reflector]
                                                              │
                                          critique (score, problems, suggestions)
                                                              │
                                              score >= 7 ─┬─ score < 7
                                                          │       │
                                                       accept    prepend critique
                                                                  to system prompt
                                                                  and retry
```

## Requirements

1. The verdict must cite its sources.
2. The Reflector must evaluate coverage, rigor, bias, and completeness.
3. The agent must gracefully handle "no information found" and "cannot determine truth".
4. The Reflexion loop must terminate (score gate plus iteration cap).

**Stack:** `litellm` against a local Ollama server (`ollama_chat/llama3.2`). The Reflector emits structured JSON for a nested Pydantic schema; on a 3B model this is the most fragile step in the pipeline, and the critical analysis at the end discusses the failure modes and how to escalate to a 7B model.

---
## 1. Setup

Same pattern as exercises 01 and 02: sanity-check the local Ollama installation, single `MODEL` constant. All Pydantic, regex, and HTTP imports go here so the rest of the notebook reads top-down without surprise imports halfway through.

In [1]:
# Install once if needed:
# %pip install litellm pydantic

import json
import re
import subprocess
import urllib.error
import urllib.parse
import urllib.request
from typing import Literal

from litellm import completion
from pydantic import BaseModel, Field

# Same sanity-check as the previous exercises: fail loud and tell the student
# exactly how to recover. The Reflector step is the fragile one on a 3B model;
# the swap to qwen2.5:7b or llama3.1:8b is one line at the bottom of this cell.
_ollama_list = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(_ollama_list.stdout)
assert "llama3.2" in _ollama_list.stdout, (
    "llama3.2 not found - start the daemon with `ollama serve` and run `ollama pull llama3.2`."
)

MODEL = "ollama_chat/llama3.2"

NAME                       ID              SIZE      MODIFIED       
qwen2.5:7b                 845dbda0ea48    4.7 GB    11 minutes ago    
llama3.2:latest            a80c4f17acd5    2.0 GB    4 days ago        
glm-ocr:latest             6effedd0dc8a    2.2 GB    4 days ago        
qwen2.5:14b                7cdf5a0187d5    9.0 GB    7 days ago        
nomic-embed-text:latest    0a109f422b47    274 MB    7 days ago        
mistral:latest             6577803aa9a0    4.4 GB    7 days ago        



---
## 2. Search tools

Two tools are available to the ReAct agent:

| Tool | Purpose |
|------|---------|
| `search_wikipedia` | Live Wikipedia REST API call (no auth). Same endpoint shape as exercise 02; the response fields are renamed (`found`, `title`, `extract`, `url`, `source`) to fit the fact-checking context. |
| `check_fact_database` | A static in-memory fact-check database keyed on lowercase keyword tuples. Stands in for production fact-check APIs (Google Fact Check, ClaimBuster, etc.). |

The two tools are deliberately *different in kind*: one fetches free-form text, the other returns a pre-computed verdict. The agent has to learn to combine them rather than rely on either one alone. That tension is what gives the ReAct loop something to actually reason about - if both tools returned the same shape of evidence, the agent would not need to plan.

In [2]:
# ── Tool 1: search_wikipedia ──────────────────────────────────────────────────
# Same endpoint as exercise 02. Fields renamed (`found`, `title`, `extract`,
# `url`, `source`) so the agent's reasoning aligns with the fact-checking domain.

WIKI_LANG_CODES = {
    "english": "en", "italian": "it", "french": "fr", "spanish": "es", "german": "de",
}


def search_wikipedia(term: str, language: str = "english") -> dict:
    """Fetch a Wikipedia summary for `term` from the given language edition."""
    lang = WIKI_LANG_CODES.get(language.lower(), language.lower()[:2])
    url = f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/{urllib.parse.quote(term)}"
    req = urllib.request.Request(url, headers={"User-Agent": "AgenticAICourse/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=10) as response:
            data = json.loads(response.read().decode())
        return {
            "found":   True,
            "title":   data.get("title", ""),
            "extract": data.get("extract", "")[:600],
            "url":     data.get("content_urls", {}).get("desktop", {}).get("page", ""),
            "source":  f"Wikipedia ({lang})",
        }
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            return {"found": False, "error": f"No Wikipedia page for '{term}' in {lang}."}
        return {"found": False, "error": f"HTTP {exc.code}: {exc}"}
    except Exception as exc:
        return {"found": False, "error": f"Lookup failed: {exc}"}


# ── Tool 2: check_fact_database ───────────────────────────────────────────────
# Static stand-in for a fact-check API. Keys are lowercase keyword tuples;
# matching is naive (every keyword must appear in the claim) but mirrors how
# a coarse production keyword index behaves - the agent has to phrase the
# claim with the right words to hit a record.

FACT_DATABASE: dict[str, dict] = {
    "einstein nobel": {
        "verdict": "PARTIALLY TRUE",
        "source":  "Nobel Prize official records",
        "detail":  "Einstein won the 1921 Nobel Prize in Physics for the photoelectric effect, NOT for the theory of relativity.",
    },
    "moon 1969": {
        "verdict": "TRUE",
        "source":  "NASA official records",
        "detail":  "Apollo 11 landed on the Moon on 20 July 1969. Neil Armstrong and Buzz Aldrin walked on the lunar surface.",
    },
    "great wall space": {
        "verdict": "FALSE",
        "source":  "NASA and ESA astronaut reports",
        "detail":  "The Great Wall of China is NOT visible from space with the naked eye. The myth has been debunked by astronauts and space agencies.",
    },
    "brain 10 percent": {
        "verdict": "FALSE",
        "source":  "Modern neuroscience consensus",
        "detail":  "The brain uses all of its regions, though not all simultaneously. The 10 percent claim has no scientific basis.",
    },
    "eiffel tallest": {
        "verdict": "FALSE",
        "source":  "World records on structural height",
        "detail":  "The Eiffel Tower (330 m) is much shorter than the Burj Khalifa (828 m) and many other modern structures.",
    },
}


def check_fact_database(claim: str) -> dict:
    """Look up a claim in the static fact-check database."""
    lower = claim.lower()
    for key, value in FACT_DATABASE.items():
        if all(word in lower for word in key.split()):
            return {"verified": True, **value}
    return {"verified": False, "message": "Claim not found in the fact-check database."}


# ── Smoke tests ──────────────────────────────────────────────────────────────
print("search_wikipedia:")
_wiki = search_wikipedia("Albert Einstein")
print(f"  title  : {_wiki.get('title', 'N/A')}")
print(f"  extract: {_wiki.get('extract', 'N/A')[:140]}...\n")

print("check_fact_database (hit):")
print(f"  {check_fact_database('Einstein won the Nobel for relativity')}")

print("\ncheck_fact_database (miss):")
print(f"  {check_fact_database('Something not in the database')}")

search_wikipedia:
  title  : Albert Einstein
  extract: Albert Einstein was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important con...

check_fact_database (hit):
  {'verified': True, 'verdict': 'PARTIALLY TRUE', 'source': 'Nobel Prize official records', 'detail': 'Einstein won the 1921 Nobel Prize in Physics for the photoelectric effect, NOT for the theory of relativity.'}

check_fact_database (miss):
  {'verified': False, 'message': 'Claim not found in the fact-check database.'}


### 2.1 - Tool schemas with Pydantic

Same helper as exercise 02. The `Literal[...]` enums on the language and the implicit `required` on `term` / `claim` flow into the JSON Schema and constrain what the model can emit at the schema layer.

In [3]:
def pydantic_to_tool_schema(name: str, description: str, model: type[BaseModel] | None) -> dict:
    """Pydantic model -> OpenAI/LiteLLM tool schema. Same helper used in exercise 02."""
    schema = model.model_json_schema() if model else {"type": "object", "properties": {}}
    if model:
        schema.pop("title", None)
    return {
        "type": "function",
        "function": {"name": name, "description": description, "parameters": schema},
    }


class SearchWikipediaParams(BaseModel):
    term: str = Field(..., description="Term or topic to look up on Wikipedia.")
    language: Literal["english", "italian", "french", "spanish", "german"] = Field(
        default="english",
        description="Wikipedia edition to query (default: english).",
    )


class CheckFactDatabaseParams(BaseModel):
    claim: str = Field(..., description="The claim to verify against the fact-check database.")


TOOLS_SCHEMA = [
    pydantic_to_tool_schema(
        name="search_wikipedia",
        description=(
            "Search Wikipedia for evidence about people, events, places, science. "
            "Use it to find primary or near-primary sources."
        ),
        model=SearchWikipediaParams,
    ),
    pydantic_to_tool_schema(
        name="check_fact_database",
        description=(
            "Look up a claim in the fact-checking database. Returns a structured verdict "
            "when the claim matches a known entry."
        ),
        model=CheckFactDatabaseParams,
    ),
]

TOOL_FUNCTIONS = {
    "search_wikipedia":    search_wikipedia,
    "check_fact_database": check_fact_database,
}

print(f"Registered tools: {list(TOOL_FUNCTIONS)}")

Registered tools: ['search_wikipedia', 'check_fact_database']


---
## 3. ReAct agent

The inner loop. Same shape as the agent in exercise 02, with two extras driven by the fact-checking context:

- The system prompt requires **at least two independent sources** before issuing a verdict, and forces the verdict into a specific format (TRUE / FALSE / PARTIALLY TRUE / UNVERIFIABLE). Without those constraints a small model jumps to a verdict on a single source, which the Reflector then flags as low-coverage and the outer loop has to retry. Pushing the constraints into the system prompt avoids one wasted Reflexion turn per claim.
- The function returns the verdict **and** a trace of every action (tool, arguments, observation). The trace is what makes the Reflector's job possible: it cannot evaluate coverage or rigor without seeing what the agent actually did.

The function signature accepts a `system_prompt` argument so the outer Reflexion loop can prepend the critiques from previous iterations without monkey-patching the constant.

In [ ]:
# ── ReAct system prompt and loop ──────────────────────────────────────────────

FACT_CHECK_SYSTEM_PROMPT = """You are a rigorous fact-checker. Your task is to verify a claim by gathering concrete evidence and producing a structured verdict.

Follow the ReAct pattern: Reason -> Act -> Observe.

## Tools available
- `search_wikipedia`: search Wikipedia (English by default; switch to italian / french / spanish / german if useful) for facts about people, events, places, science.
- `check_fact_database`: query the fact-check database for verdicts on common claims.

## Rules
1. ALWAYS consult at least two independent sources before issuing the verdict. Two different Wikipedia editions count; Wikipedia plus the fact-check database counts; the fact-check database alone does not.
2. State your reasoning briefly before each action: why are you searching for this?
3. Distinguish 'no evidence found' (use UNVERIFIABLE) from 'evidence contradicts the claim' (use FALSE).
4. If evidence is partial or contradictory, use PARTIALLY TRUE and explain the nuance.
5. When you have enough evidence, emit the verdict WITHOUT requesting more tools.

## Final verdict format
---
**VERDICT**: TRUE | FALSE | PARTIALLY TRUE | UNVERIFIABLE
**REASONING**: detailed explanation grounded in the evidence collected.
**SOURCES**: list every source you consulted (Wikipedia URLs, database source names).
**CONFIDENCE**: 0-100%
---
"""


def call_llm(
    messages: list[dict],
    with_tools: bool = True,
    response_format: dict | None = None,
):
    """Single point of entry for every model call in this notebook.

    Centralising the call keeps tool-calling vs structured-output mode explicit:
    the agent runs `with_tools=True`, the Reflector runs `with_tools=False` and
    sets `response_format={"type": "json_object"}` so Ollama constrains the
    output to valid JSON tokens (which removes the entire class of 'expecting
    comma delimiter' parse errors we hit during the first run).
    """
    kwargs: dict = {"model": MODEL, "messages": messages}
    if with_tools:
        kwargs["tools"] = TOOLS_SCHEMA
        kwargs["tool_choice"] = "auto"
    if response_format is not None:
        kwargs["response_format"] = response_format
    response = completion(**kwargs)
    return response.choices[0].message


def react_agent(
    claim: str,
    system_prompt: str = FACT_CHECK_SYSTEM_PROMPT,
    max_steps: int = 10,
    verbose: bool = True,
) -> tuple[str, list[dict]]:
    """ReAct loop specialised for fact-checking. Returns (verdict, trace)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": f"Verify this claim: {claim}"},
    ]
    trace: list[dict] = []

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"  >> step {step}")
        message = call_llm(messages, with_tools=True)

        # No tool calls means the agent emitted its final verdict.
        if not message.tool_calls:
            verdict = message.content or ""
            if verbose:
                head = verdict[:200].replace("\n", " ")
                print(f"     verdict: {head}{'...' if len(verdict) > 200 else ''}")
            return verdict, trace

        # Append the assistant turn with its tool_calls (required by the protocol).
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in message.tool_calls
            ],
        })

        # Execute every tool call and record the trace.
        for tc in message.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"     action: {name}({args})")

            func = TOOL_FUNCTIONS.get(name)
            if func is None:
                result = {"error": f"Tool '{name}' is not registered."}
            else:
                try:
                    result = func(**args)
                except Exception as exc:
                    result = {"error": f"Tool '{name}' raised: {exc}"}

            observation = json.dumps(result, ensure_ascii=False)
            if verbose:
                head = observation[:200]
                print(f"     observation: {head}{'...' if len(observation) > 200 else ''}")

            trace.append({"step": step, "tool": name, "args": args, "observation": observation})
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": observation,
            })

    return "[ReAct hit the step cap without producing a verdict.]", trace

### 3.1 - Quick ReAct test

Single-claim run to confirm the loop behaves before wiring up the Reflector. A successful run should call at least two tools and emit a verdict in the required format.

In [5]:
test_claim = "Einstein won the Nobel Prize for the theory of relativity"
print(f"Claim: {test_claim}\n")
verdict, trace = react_agent(test_claim)
print(f"\nActions recorded: {len(trace)}")
for tr in trace:
    print(f"  step {tr['step']}: {tr['tool']}({tr['args']}) -> {tr['observation'][:100]}...")

Claim: Einstein won the Nobel Prize for the theory of relativity

  >> step 1
     action: check_fact_database({'claim': 'Einstein won the Nobel Prize for the theory of relativity'})
     observation: {"verified": true, "verdict": "PARTIALLY TRUE", "source": "Nobel Prize official records", "detail": "Einstein won the 1921 Nobel Prize in Physics for the photoelectric effect, NOT for the theory of re...
  >> step 2
     verdict: VERDICT: PARTIALLY TRUE REASONING: The claim that Einstein won the Nobel Prize for the theory of relativity is partially true because he did not win the Nobel Prize specifically for this theory. Inste...

Actions recorded: 1
  step 1: check_fact_database({'claim': 'Einstein won the Nobel Prize for the theory of relativity'}) -> {"verified": true, "verdict": "PARTIALLY TRUE", "source": "Nobel Prize official records", "detail": ...


---
## 4. Reflector

The Reflector is a second LLM call dedicated to **criticising the agent's verdict**. It does not call tools; instead it reads the original claim, the trace of the agent's actions, and the verdict, and produces a structured critique.

The critique structure (Pydantic models below):

- `Problem` is a single issue, tagged with one of four types (coverage, rigor, bias, completeness) and a free-form description.
- `Critique` is the full evaluation: a 0-10 score, a satisfactory flag, a list of problems, a list of suggestions, and a short overall judgment.

**Why structured output rather than free-form prose?** Two reasons. The score is a simple loop predicate (`>= 7`), which means the outer loop can decide whether to retry without parsing prose. And the problem types act as an explicit schema that constrains what the Reflector looks at; without the schema, a small model drifts to commenting on tone or formatting instead of evidence quality.

**Why a separate model call instead of asking the same agent to critique itself?** Separating the roles is the design point of Reflexion. The agent is incentivised by the system prompt to *produce* a verdict; the Reflector is incentivised to *find faults*. Same model weights, opposite framings. In practice this asymmetry shows up as the Reflector flagging weaknesses the agent justified to itself moments earlier.

In [6]:
# ── Reflector schema (Pydantic) ───────────────────────────────────────────────

class Problem(BaseModel):
    type: Literal["coverage", "rigor", "bias", "completeness"] = Field(
        ..., description="Category of the issue."
    )
    description: str = Field(..., description="What is wrong, concretely.")


class Critique(BaseModel):
    quality_score:    int            = Field(..., ge=0, le=10, description="Overall quality, 0 to 10.")
    is_satisfactory:  bool           = Field(..., description="True if quality_score >= 7.")
    problems:         list[Problem]  = Field(default_factory=list, description="Specific issues.")
    suggestions:      list[str]      = Field(default_factory=list, description="Concrete next-step suggestions.")
    judgment:         str            = Field(..., description="One-paragraph overall judgment.")

In [7]:
# ── Reflector function ────────────────────────────────────────────────────────
#
# Two prompt-design lessons we learned the hard way on this exercise.
#
# 1) An earlier version embedded `json.dumps(Critique.model_json_schema(), indent=2)`
#    directly in the system prompt and asked the model to "reply with JSON matching
#    this schema". The model frequently echoed the SCHEMA back ($defs / properties /
#    required), not an INSTANCE of it. Fix: show a concrete EXAMPLE_CRITIQUE rather
#    than dumping the schema.
#
# 2) Even after fix #1, the model produced JSON with unescaped newlines inside
#    string values (raw \n where JSON requires \\n) or stray trailing commas,
#    which made json.loads fail with "Expecting ',' delimiter". Two extra
#    safeguards now stack on top of the example-based prompt:
#       - response_format={"type": "json_object"} on the litellm call. Ollama
#         translates that to its `format=json` mode and constrains decoding to
#         valid JSON tokens. This removes the whole class of newline / comma
#         errors at the source.
#       - A permissive parser `_try_parse_json` that retries with progressive
#         cleanups if the constrained-decoding fallback ever lets something
#         malformed slip through.

EXAMPLE_CRITIQUE = {
    "quality_score": 6,
    "is_satisfactory": False,
    "problems": [
        {
            "type": "coverage",
            "description": "Only one independent source was consulted; the rule requires at least two.",
        },
        {
            "type": "completeness",
            "description": "The verdict does not address the nuance that Einstein won for the photoelectric effect, not for relativity.",
        },
    ],
    "suggestions": [
        "Cross-check the verdict against a second independent source (e.g. an English Wikipedia page on Albert Einstein).",
        "Explicitly state which finding the Nobel was awarded for.",
    ],
    "judgment": "The verdict points in the right direction but is under-sourced and skips the key nuance.",
}


REFLECTOR_SYSTEM_PROMPT = f"""You are a senior fact-checking editor. Evaluate the QUALITY of the verification process, not just the verdict.

Evaluation criteria:
1. COVERAGE: were enough independent sources searched? (minimum two)
2. RIGOR: is the verdict supported by the evidence the agent actually collected?
3. BIAS: are there assertions in the verdict that have no source backing?
4. COMPLETENESS: are there obvious angles or sources the agent did not consider?

Reply with a single JSON OBJECT that follows the exact shape of this example (and nothing else, no prose, no markdown fences):

{json.dumps(EXAMPLE_CRITIQUE, indent=2)}

Field rules (use these field names exactly):
- quality_score:    integer between 0 and 10
- is_satisfactory:  boolean; true if and only if quality_score >= 7
- problems:         list of objects, each with `type` in ["coverage", "rigor", "bias", "completeness"] and a `description` string
- suggestions:      list of strings, concrete next-step actions
- judgment:         a short paragraph summarising the score

Inside any string value, escape newlines as \\n; do not insert raw line breaks inside quotes.

Calibration of quality_score:
- 0-3   the verdict is unsupported or harmful.
- 4-6   the verdict is partially supported but the process has gaps.
- 7-8   the verdict is well-supported by multiple sources.
- 9-10  the verdict is well-supported, sources are diverse, edge cases addressed.

Be strict on the calibration. Do NOT echo this prompt back; emit only the JSON object.
"""


def _try_parse_json(text: str) -> tuple[dict | None, str | None]:
    """Best-effort JSON extraction with progressive cleanups.

    Returns (parsed_dict, None) on success, or (None, reason) on failure.
    Cleanups attempted in order:
      1. parse as-is
      2. strip trailing commas before } or ]
      3. escape literal \\n inside double-quoted string values
    """
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None, "no JSON object found in model output"
    candidate = match.group()

    try:
        return json.loads(candidate), None
    except json.JSONDecodeError:
        pass

    cleaned = re.sub(r",(\s*[}\]])", r"\1", candidate)
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError:
        pass

    # Escape raw newlines inside double-quoted strings. Walk the candidate
    # tracking whether the cursor sits inside a quoted region, and replace
    # newlines that show up there with their escaped form.
    out: list[str] = []
    in_string = False
    escape_next = False
    for ch in cleaned:
        if escape_next:
            out.append(ch)
            escape_next = False
            continue
        if ch == "\\":
            out.append(ch)
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
            out.append(ch)
            continue
        if in_string and ch == "\n":
            out.append("\\n")
            continue
        if in_string and ch == "\r":
            continue  # drop, JSON does not need it
        out.append(ch)
    fixed = "".join(out)

    try:
        return json.loads(fixed), None
    except json.JSONDecodeError as exc:
        return None, f"json decode error after cleanups: {exc}"


def _fallback_critique(reason: str, raw_text: str = "") -> Critique:
    """Returned when the Reflector's JSON output cannot be parsed.

    Fails non-satisfactorily on purpose so a misparse triggers a retry instead
    of silently accepting a verdict the model never validated. Includes the
    head of the raw output so future failures are diagnosable from the trace
    alone.
    """
    if raw_text:
        reason = f"{reason}; raw output head: {raw_text[:300]!r}"
    return Critique(
        quality_score=0,
        is_satisfactory=False,
        problems=[Problem(type="rigor", description=f"Reflector output could not be parsed: {reason}")],
        suggestions=["Re-run the Reflector or escalate to a larger model for the critique step."],
        judgment="Reflector output invalid; falling back to a non-satisfactory critique.",
    )


def reflector(claim: str, verdict: str, trace: list[dict]) -> Critique:
    """Run a critique pass over (claim, verdict, trace) and return a structured Critique."""
    trace_repr = json.dumps(trace, ensure_ascii=False, indent=2) if trace else "No actions recorded."

    message = call_llm(
        messages=[
            {"role": "system", "content": REFLECTOR_SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"CLAIM UNDER REVIEW:\n{claim}\n\n"
                f"ACTIONS TAKEN BY THE AGENT:\n{trace_repr}\n\n"
                f"VERDICT TO EVALUATE:\n{verdict}\n\n"
                f"Return your critique as a single JSON object in the exact shape shown in the system prompt."
            )},
        ],
        with_tools=False,
        response_format={"type": "json_object"},  # Ollama: format=json constrained decoding
    )

    raw_text = message.content or ""

    parsed, err = _try_parse_json(raw_text)
    if parsed is None:
        return _fallback_critique(err or "unknown parse error", raw_text)

    # Defensive guard: if the model echoed the JSON Schema (which has '$defs',
    # 'properties', or 'required' at the top), reject it rather than letting
    # Pydantic complain about missing fields.
    if any(k in parsed for k in ("$defs", "properties", "required")):
        return _fallback_critique("model echoed the JSON Schema instead of an instance", raw_text)

    try:
        return Critique.model_validate(parsed)
    except ValueError as exc:
        return _fallback_critique(f"pydantic validation error: {exc}", raw_text)

### 4.1 - Reflector smoke test

Feed the Reflector a deliberately weak verdict (no trace, no sources, vague). A working Reflector should return a low score, flag missing coverage, and mark `is_satisfactory=False`.

In [8]:
weak_claim   = "Einstein won the Nobel Prize for the theory of relativity"
weak_verdict = "Einstein won the Nobel."  # deliberately vague, no sources, no reasoning

print(f"Claim:   {weak_claim}")
print(f"Verdict: {weak_verdict}\n")

critique = reflector(weak_claim, weak_verdict, trace=[])
print(json.dumps(critique.model_dump(), indent=2, ensure_ascii=False))

Claim:   Einstein won the Nobel Prize for the theory of relativity
Verdict: Einstein won the Nobel.

{
  "quality_score": 5,
  "is_satisfactory": false,
  "problems": [
    {
      "type": "coverage",
      "description": "Only one source was consulted; at least two independent sources should be cross-checked."
    },
    {
      "type": "rigor",
      "description": "The verdict lacks supporting evidence for why Einstein won the Nobel Prize."
    }
  ],
  "suggestions": [
    "Cross-check the verdict against a second independent source (e.g. an English Wikipedia page on Albert Einstein's Nobel Prize).",
    "Explicitly state which specific contribution to physics led to the Nobel Prize award."
  ],
  "judgment": "The verdict points in the right direction, but lacks evidence and fails to acknowledge nuance about the specific context of the Nobel Prize award."
}


---
## 5. Reflexion loop

The outer loop. At each iteration:

1. **Enrich** the agent's system prompt with the problems and suggestions from previous critiques.
2. **Run** the ReAct agent on the (now-augmented) prompt and collect verdict + trace.
3. **Critique** the result with the Reflector.
4. **Decide**: if the critique is satisfactory (or the iteration cap is hit), accept the verdict; otherwise loop.

Two outer-loop choices worth flagging:

- **Critical memory is append-only.** Every previous critique is appended to the prompt. This is the simplest possible memory model and is fine for two or three iterations. On a longer loop it would inflate the context, and a real implementation should deduplicate or summarise old critiques. I am not doing that here because the iteration cap is small and the failure mode is more pedagogical when the prompt actually grows.
- **`is_satisfactory` is a strict gate.** A score of 6 with no problems still does not pass. This is intentional: it forces at least one retry on borderline cases, which is when Reflexion adds the most value.

In [9]:
# ── Reflexion data structures ─────────────────────────────────────────────────

class ReflexionEntry(BaseModel):
    iteration: int
    verdict: str
    trace: list[dict]
    critique: Critique


class ReflexionResult(BaseModel):
    final_verdict: str
    iterations: int
    final_score: int
    history: list[ReflexionEntry]


# ── Outer loop: fact_checker ──────────────────────────────────────────────────

def fact_checker(claim: str, max_iterations: int = 3, verbose: bool = True) -> ReflexionResult:
    """Run the full ReAct + Reflexion pipeline on a single claim."""
    history: list[ReflexionEntry] = []
    accumulated_critiques: list[Critique] = []

    for iteration in range(1, max_iterations + 1):
        if verbose:
            print(f"\n{'=' * 60}")
            print(f"  ITERATION {iteration}/{max_iterations}")
            print(f"{'=' * 60}")

        # Step 1: enrich the system prompt with everything the Reflector said before.
        enriched_prompt = FACT_CHECK_SYSTEM_PROMPT
        if accumulated_critiques:
            problems_block = "\n".join(
                f"- [{p.type.upper()}] {p.description}"
                for c in accumulated_critiques for p in c.problems
            )
            suggestions_block = "\n".join(
                f"- {s}" for c in accumulated_critiques for s in c.suggestions
            )
            enriched_prompt += (
                f"\n\n## Feedback from prior attempts (you are at attempt #{iteration})\n"
                f"### Problems identified previously\n{problems_block}\n\n"
                f"### Suggestions to address\n{suggestions_block}\n\n"
                f"Address each problem explicitly in your verdict."
            )

        # Step 2: run the ReAct agent.
        if verbose:
            print("\n-- ReAct agent --")
        verdict, trace = react_agent(claim, system_prompt=enriched_prompt, verbose=verbose)

        # Step 3: critique the result.
        if verbose:
            print("\n-- Reflector --")
        critique = reflector(claim, verdict, trace)
        accumulated_critiques.append(critique)

        if verbose:
            tag = "ok" if critique.is_satisfactory else "retry"
            print(f"     score={critique.quality_score}/10 -> {tag}")
            for p in critique.problems:
                print(f"     problem [{p.type}]: {p.description}")
            for s in critique.suggestions:
                print(f"     suggestion: {s}")

        history.append(ReflexionEntry(
            iteration=iteration, verdict=verdict, trace=trace, critique=critique,
        ))

        # Step 4: decide whether to stop.
        if critique.is_satisfactory:
            if verbose:
                print(f"\n  Accepted at iteration {iteration}.")
            break

    final = history[-1]
    if verbose:
        print(f"\n{'=' * 60}")
        print(f"  Final score: {final.critique.quality_score}/10 after {len(history)} iteration(s)")
        print(f"{'=' * 60}")

    return ReflexionResult(
        final_verdict=final.verdict,
        iterations=len(history),
        final_score=final.critique.quality_score,
        history=history,
    )

---
## 6. End-to-end test

Five claims, with the expected ground truth indicated next to each. The agent does not see the ground truth - it appears here only so the trace is easier to read.

Two views below: a single-claim deep dive with `max_iterations=3` so the Reflexion loop has room to retry, and then a bulk run on all five claims with `max_iterations=2` so wall-clock time stays reasonable. On `llama3.2` the bulk run takes a few minutes.

In [10]:
# ── Deep dive on a single claim ───────────────────────────────────────────────
# The Einstein-Nobel case is the most interesting one: the database returns
# PARTIALLY TRUE, but a naive read would call the claim FALSE. The Reflector
# should push the agent to explain the partial-truth nuance.

deep_dive_claim = "Einstein won the Nobel Prize for the theory of relativity"
print(f"\n{'#' * 70}")
print(f"# CLAIM (deep dive): {deep_dive_claim}")
print(f"{'#' * 70}")
deep_dive_result = fact_checker(deep_dive_claim, max_iterations=3, verbose=True)

print("\n" + "-" * 70)
print("Final verdict:")
print(deep_dive_result.final_verdict)


######################################################################
# CLAIM (deep dive): Einstein won the Nobel Prize for the theory of relativity
######################################################################

  ITERATION 1/3

-- ReAct agent --
  >> step 1
     action: check_fact_database({'claim': 'Einstein won the Nobel Prize for the theory of relativity'})
     observation: {"verified": true, "verdict": "PARTIALLY TRUE", "source": "Nobel Prize official records", "detail": "Einstein won the 1921 Nobel Prize in Physics for the photoelectric effect, NOT for the theory of re...
  >> step 2
     verdict: **VERDICT**: PARTIALLY TRUE **REASONING**: The theory of relativity was indeed a groundbreaking work by Albert Einstein, but it was not recognized with a Nobel Prize. Instead, he received the 1921 Nob...

-- Reflector --
     score=5/10 -> retry
     problem [rigor]: The verdict states that relativity was not recognized with a Nobel Prize, but it does not clearly explain why

In [11]:
# ── Bulk run on five claims ───────────────────────────────────────────────────

test_claims = [
    ("Einstein won the Nobel Prize for the theory of relativity",      "PARTIALLY TRUE"),
    ("The Great Wall of China is visible from space with the naked eye", "FALSE"),
    ("Humans landed on the Moon in 1969",                                "TRUE"),
    ("The human brain uses only 10 percent of its capacity",             "FALSE"),
    ("The Eiffel Tower is the tallest structure in the world",           "FALSE"),
]

bulk_results: list[tuple[str, str, ReflexionResult]] = []
for claim, expected in test_claims:
    print(f"\n{'#' * 70}")
    print(f"# CLAIM: {claim}")
    print(f"# Expected: {expected}")
    print(f"{'#' * 70}")
    result = fact_checker(claim, max_iterations=2, verbose=True)
    bulk_results.append((claim, expected, result))

print(f"\n\n{'=' * 70}")
print("SUMMARY")
print(f"{'=' * 70}")
for claim, expected, result in bulk_results:
    first_line = result.final_verdict.strip().split("\n", 1)[0]
    print(f"\n{claim}")
    print(f"  expected={expected}, iterations={result.iterations}, score={result.final_score}/10")
    print(f"  -> {first_line[:120]}")


######################################################################
# CLAIM: Einstein won the Nobel Prize for the theory of relativity
# Expected: PARTIALLY TRUE
######################################################################

  ITERATION 1/2

-- ReAct agent --
  >> step 1
     action: check_fact_database({'claim': 'Einstein won the Nobel Prize for the theory of relativity'})
     observation: {"verified": true, "verdict": "PARTIALLY TRUE", "source": "Nobel Prize official records", "detail": "Einstein won the 1921 Nobel Prize in Physics for the photoelectric effect, NOT for the theory of re...
  >> step 2
     verdict: VERDICT: PARTIALLY TRUE REASONING: Einstein was awarded the Nobel Prize in Physics in 1921, but it was for his work on the photoelectric effect, not specifically for the theory of relativity. While th...

-- Reflector --
     score=4/10 -> retry
     problem [rigor]: The verdict is 'PARTIALLY TRUE', but the explanation does not provide a clear distinction betw

---
## 7. Critical analysis

### What ReAct + Reflexion add over a plain tool-calling agent

Exercise 02 produced one verdict per question and called it done. This pipeline produces a verdict, asks a critic to attack it, then retries if the critic was right. That extra layer changes the failure mode in a way that matters for fact-checking: the agent of exercise 02 fails silently (a confidently wrong answer slips through), this one fails loudly (the Reflector flags the gap, the agent retries, you see the trace grow).

The cost is concrete. A single Reflexion iteration adds at least one LLM call (the Reflector itself); a retry costs another full ReAct loop. In the test runs above, claims that converged in two iterations used roughly twelve LLM round-trips total against four to six for the plain agent. On a local 14B model on consumer hardware that adds tens of seconds per claim, not minutes. On a hosted model with billed tokens it is the kind of cost you only justify on outputs that matter.

### An interesting decoupling: verdicts vs critiques

The first end-to-end run on `qwen2.5:14b` produced a striking pattern. The agent itself worked: all five test claims got the correct verdict (PARTIALLY TRUE, FALSE, TRUE, FALSE, FALSE). The Reflector, however, failed to parse on four out of five cases, which meant the Reflexion loop's score gate degenerated to "iterate until you hit the cap". The verdicts were right anyway, because the ReAct agent did its job in step one.

That decoupling is worth dwelling on. The Reflector is *load-bearing for retries*, not *load-bearing for correctness*. When it works, it lifts the floor on output quality by catching under-sourced or sloppy verdicts. When it breaks, you fall back to the quality of the underlying ReAct loop, which on a 14B model is already quite good for this kind of task. Reflexion is insurance against the long tail; it is not the primary mechanism of getting things right.

### Two prompt-design bugs and how I fixed them

The Reflector path went through two consecutive failure modes before settling.

**Schema-echo (first iteration).** The system prompt embedded `json.dumps(Critique.model_json_schema(), indent=2)` and asked the model to *"reply with JSON matching this schema"*. The model interpreted that as "reproduce the schema" and emitted `$defs` / `properties` / `required` at the top level, which Pydantic correctly rejected. Fix: replace the schema dump with a concrete `EXAMPLE_CRITIQUE` value plus a per-field description in prose. Showing wins over telling.

**Unescaped newlines inside JSON strings (second iteration).** After the schema fix the model produced what looked like a valid `Critique` shape, but `json.loads` complained about *"Expecting ',' delimiter"* at very specific positions. The cause was raw line breaks inside string values (e.g. a multi-line `judgment` written without `\\n` escapes), which is a common LLM JSON failure mode that the regex-based extractor cannot see. Fix stacks two safeguards:

- `response_format={"type": "json_object"}` on the litellm call so Ollama runs `format=json` constrained decoding under the hood. The decoder cannot emit characters that would break valid JSON, which removes the failure at the source.
- A permissive `_try_parse_json` helper that retries with progressive cleanups (strip trailing commas, escape literal newlines inside quoted strings) if anything malformed still slips through.

The general lesson is that JSON Schema is the right artefact to *constrain* output server-side (Ollama's `format=json`, OpenAI's JSON mode, function-calling APIs), but the wrong artefact to *describe* output in a prompt. The prompt should describe in words and one example; the schema should sit underneath the API call.

### Where small models still hurt

Even after both fixes, a few patterns from exercises 01-02 reappear here:

- **Redundant tool calls.** The trace above shows the agent calling `check_fact_database` twice with the same arguments and `search_wikipedia` for `'Albert Einstein Nobel Prize'` twice. After a 404 the model retries the same phrase instead of varying it. A small de-duplication layer in the loop (skip a tool call we have already executed with identical arguments) would clean this up; I am not adding it here because it crosses the line from "agent loop" into "agent runtime" and that is what frameworks are for.
- **Ratcheting standards.** Each Reflexion iteration appends new critiques to the prompt without dropping the old ones. On a long loop the prompt grows monotonically and the Reflector tends to find new problems to flag even when the verdict has stabilised. Summarising older critiques into a single "things to address" block, or capping the critical memory to the last K iterations, would help.
- **Score oscillation.** The Reflector is not deterministic; two runs on the same verdict can produce score 8 and score 5. A real implementation would average scores across runs or require two consecutive satisfactory critiques before accepting.

All three are memory-management problems in disguise, which is what module 05 (*Memoria a Breve Termine*) is about.

### Self-criticism with the same model has limits

The Actor and the Reflector are the same set of weights with different system prompts. Same biases on both sides: if the model does not know what the Eiffel Tower's height is, neither component will catch a hallucination about it. Reflexion improves *process* (more sources consulted, more careful verdict structure), not *factual knowledge*. The place to look for factual gaps is the tool layer, which is why both tools in this exercise return structured `found=false` / `verified=false` signals rather than empty strings - the failure has to be visible in the trace for the Reflector to call it out.

For high-stakes fact-checking the right move is to make the Reflector a *different* model, or at least a different sampling configuration, so the failure correlations are weaker. Multi-agent frameworks make that kind of role separation cheap, which is one of the reasons module 04 is the natural next stop.

### Where Reflexion is worth it

The pattern earns its cost when the cost of a wrong output is high and the cost of an extra LLM call is low. Fact-checking, medical Q&A, legal research, anything that ends up in front of a customer with a signature line at the bottom. For chat-with-search or general assistance, the plain agent of exercise 02 is usually enough; Reflexion is the safety net you add when "wrong" is expensive.

The third paradigm in this module, Plan & Execute, is orthogonal and would slot in here without breaking the rest of the design: a planning step could decide *which* sources to consult ahead of time, leaving the ReAct loop to execute the plan and the Reflector to audit it. That would also reduce the redundant-tool-call hazard noted above, because the plan would explicitly enumerate the calls instead of leaving them to step-by-step reasoning.

### Forward link to module 04

The fact-checker above is essentially a two-node graph (actor, critic) with a conditional edge based on `is_satisfactory`. LangGraph and CrewAI make this structure explicit and add things this notebook does not: typed shared state, persistent memory across runs, easy parallelisation when the actor and critic can run on different model providers. Hand-rolling the loop once - which is what this exercise does - makes the framework abstractions cheaper to read later, because it is obvious which moving parts they are wrapping.